# Embedding, vector index and search (KAN-97)
Gözde Nur Özdemir · ELIZA · Sprint 2

**Goal:** turn the 105 chunks from `src/chunking.ipynb` into vectors, store them in a vector database, and return the most relevant chunks for a question. This is the course's Sprint 2 baseline (*"the most basic vector database and retrieval over your documents"*) and the halfway check on 11/11 (*"retrieval returns passages for a handful of test questions"*).

**Rule of this notebook: every choice has a reason that can be checked.** A choice is either taken from a source (the design document, a model card, a paper, the instructor's slides) or measured here on our own questions. Nothing is removed or set "because it is usual".

| Choice | Where the reason comes from |
|---|---|
| Embedding model `BAAI/bge-base-en-v1.5` | Design decision D10 (model selection spike) |
| Instruction in front of questions | Model card of bge-v1.5, **checked in step 5** |
| Full text for the embedding model, no stopword removal | **Measured in step 5** |
| Which words BM25 ignores | Frequency in our chunks (step 2) + LLM judge (step 3) + **measured in step 4** |
| Chroma, cosine distance | Design decision D11 |
| Hybrid search with Reciprocal Rank Fusion, k = 60 | Design decision D12; k = 60 from Cormack et al. 2009 (the RRF paper); **measured in step 7** |
| Top-k | Design decision D12 (choose 3, 5 or 8 by Recall@k), **measured in step 8** |
| How we score retrieval: Recall@k, Hit@k, MRR | Instructor's slides: *"most use an LLM judge, so also compute Recall@k / MRR on gold IDs"* |

**Steps**
1. Load the settings, the chunks and the development questions
2. Which words are in most chunks?
3. LLM judge: which frequent words are useless for search?
4. Keyword search (BM25): which words to ignore, measured
5. Embedding: full text or without stopwords, with or without the instruction, measured
6. Store the vectors in Chroma
7. Dense vs BM25 vs hybrid, measured
8. How many chunks to return (top-k), measured
9. A handful of test questions (11/11 check)
10. The decisions and their evidence

**How to run:** open this notebook from the `src/` folder. Step 3 needs [Ollama](https://ollama.com) with the judge model: `ollama pull qwen3:4b`. The first run also downloads the embedding model (about 440 MB). A normal laptop is enough; no GPU.

In [ ]:
# Install libraries
%pip -q install sentence-transformers chromadb rank_bm25 pandas pyyaml requests scikit-learn

import csv, json, re, time
from collections import Counter
import numpy as np
import pandas as pd
import requests
import yaml
pd.set_option("display.max_colwidth", None)

## 1. Load the settings, the chunks and the development questions
**Development questions:** we tune only on the six Annex III 3(b) questions **B-01 to B-06** (KAN-46), because they are planned for the `val` split. The other questions are not used here, so they can still be a fair test later (the `val`/`test` split is decided in KAN-53). Six questions are few, so the results here are a first check; the final numbers come from the full `val` split in KAN-99.

**Scoring** (instructor's slides): for each question, the gold sources are the provisions the team marked as the answer.
- **Recall@k:** share of the gold provisions found in the top k chunks, averaged over the questions
- **Hit@k:** share of questions with at least one gold provision in the top k
- **MRR:** 1 / rank of the first correct chunk, averaged (1.0 = always first)

In [ ]:
CONFIG = yaml.safe_load(open("../config/rag.yaml"))
EMB, STORE, RET, BM, JUDGE = (CONFIG[k] for k in ["embedding", "vector_store", "retrieval", "bm25", "judge"])

chunks = [json.loads(line) for line in open("../data/chunks.jsonl", encoding="utf-8")]
ref_of = {c["chunk_id"]: c["ref"] for c in chunks}

all_questions = {r["id"]: r for r in csv.DictReader(open("../docs/eval/eval-set-v1.csv", encoding="utf-8-sig"))}
dev = [all_questions[q] for q in ["B-01", "B-02", "B-03", "B-04", "B-05", "B-06"]]
print(len(chunks), "chunks;", len(dev), "development questions")
pd.DataFrame(dev)[["id", "question", "gold_source"]]

In [ ]:
def gold_parts(question):
    """The gold provisions of a question, e.g. ['Article 6(2)', 'Article 6(3), first subparagraph']."""
    return [p.strip() for p in question["gold_source"].split(";") if p.strip()]

def same_provision(ref, gold):
    """True if a chunk reference and a gold reference are the same provision, or one is part of the other."""
    return ref.startswith(gold) or gold.startswith(ref)

def evaluate(search_fn, k, questions=dev):
    """Recall@k, Hit@k and MRR of a search function (question text -> ranked chunk IDs)."""
    recall, hit, rr = [], [], []
    for q in questions:
        refs = [ref_of[i] for i in search_fn(q["question"])[:k]]
        gold = gold_parts(q)
        found = [g for g in gold if any(same_provision(r, g) for r in refs)]
        recall.append(len(found) / len(gold))
        hit.append(1 if found else 0)
        first = next((rank for rank, r in enumerate(refs, 1) if any(same_provision(r, g) for g in gold)), None)
        rr.append(1 / first if first else 0)
    return {"k": k, "Recall@k": round(np.mean(recall), 3), "Hit@k": round(np.mean(hit), 3), "MRR": round(np.mean(rr), 3)}

## 2. Which words are in most chunks?
We count, for every word, the share of the 105 chunks it appears in. This is only the **measurement**; it does not decide anything. A frequent word is a **candidate** to ignore, because a word that is in almost every chunk does not help to tell the chunks apart.

We also count how often the same word is in **questions**. A word that is in many chunks but also in many questions may still matter for search.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def words_of(text):
    """Lower case words (letters and numbers)."""
    return re.findall(r"[a-z0-9]+", text.lower())

def share(word_lists, word):
    return sum(word in set(ws) for ws in word_lists) / len(word_lists)

chunk_words = [words_of(c["text"]) for c in chunks]
question_words = [words_of(q["question"]) for q in all_questions.values()]
counts = Counter(w for ws in chunk_words for w in set(ws))

freq = pd.DataFrame([{"word": w, "share of chunks": n / len(chunks), "share of questions": share(question_words, w)}
                     for w, n in counts.items()]).sort_values("share of chunks", ascending=False)
frequent = freq[freq["share of chunks"] >= BM["candidate_min_share"]]
candidates = frequent[~frequent["word"].isin(ENGLISH_STOP_WORDS)]     # stopwords are tested separately in step 4
print(len(frequent), "words are in at least", f"{BM['candidate_min_share']:.0%}", "of the chunks;",
      len(candidates), "of them are not standard stopwords and go to the LLM judge")
candidates.round(2).reset_index(drop=True)

**Comment:** _(1–2 sentences after running: which frequent words are also in many questions?)_

## 3. LLM judge: which frequent words are useless for search?
Deciding by frequency alone would remove words that carry meaning (the instructor's feedback: no automatic TF-IDF cut, the context would be lost). So an LLM reads each candidate **with examples from our chunks** and answers **KEEP** or **IGNORE** with a reason.

- **Judge model:** `qwen3:4b` through Ollama, thinking turned off. It is from a **different model family** than the answer model (Llama-3.1-8B, D9), as the instructor advised for LLM judges. Qwen3-4B answered correctly with thinking off in the model selection spike.
- **Temperature 0**, so the labels can be repeated.
- **Human check:** the LLM only proposes. A team member reads the table and can change any label in `REVIEW` below. Only the final list is used in step 4.

In [ ]:
PROMPT = """You help build a keyword search over EU AI Act provisions for questions from EdTech developers.
The word below appears in {share:.0%} of the indexed provisions. Examples:
{examples}

Should keyword search IGNORE this word, because it is common wording that does not help to tell the
provisions apart, or KEEP it, because a question can depend on it (it names a legal concept, actor,
obligation, condition, negation or date)? If in doubt, answer KEEP.
Reply in one line: IGNORE or KEEP, a colon, and a reason of at most 15 words.
WORD: "{word}\""""

def examples(word, n=3):
    """Up to n short passages from the chunks that contain the word."""
    out = []
    for c in chunks:
        m = re.search(r"\b" + re.escape(word) + r"\b", c["text"], re.I)
        if m:
            out.append(f'- {c["ref"]}: "...{c["text"][max(0, m.start() - 70):m.end() + 70]}..."')
        if len(out) == n:
            break
    return "\n".join(out)

def ask_judge(prompt):
    """One answer from the judge model in Ollama (temperature 0, thinking off)."""
    r = requests.post(JUDGE["url"], json={"model": JUDGE["model"], "messages": [{"role": "user", "content": prompt}],
                                          "think": False, "stream": False, "options": {"temperature": 0}}, timeout=300)
    return r.json()["message"]["content"].strip()

labels = []
for _, row in candidates.iterrows():
    answer = ask_judge(PROMPT.format(share=row["share of chunks"], examples=examples(row["word"]), word=row["word"]))
    m = re.match(r"\W*(IGNORE|KEEP)\W*(.*)", answer, re.I | re.S)
    labels.append({"word": row["word"], "share of chunks": round(row["share of chunks"], 2),
                   "share of questions": round(row["share of questions"], 2),
                   "LLM label": m[1].upper() if m else "UNCLEAR", "LLM reason": (m[2] if m else answer).split("\n")[0]})
judge_table = pd.DataFrame(labels)
judge_table

**Human review.** Read the table. If you disagree with a label, add the word here with your decision, for example `{"risk": "KEEP"}`. Leave it empty if you agree with all labels.

In [ ]:
REVIEW = {}          # your corrections, e.g. {"risk": "KEEP"}
REVIEWER = "Gözde Nur Özdemir"

judge_table["final"] = [REVIEW.get(w, label) for w, label in zip(judge_table["word"], judge_table["LLM label"])]
judge_table["reviewer"] = REVIEWER
LLM_IGNORE = set(judge_table.loc[judge_table["final"] == "IGNORE", "word"])
print("LLM and reviewer disagree on:", sorted(REVIEW) or "none")
print("Words to ignore after review:", sorted(LLM_IGNORE))

**Comment:** _(1–2 sentences after running: do the LLM's reasons make sense? which labels did you change and why?)_

## 4. Keyword search (BM25): which words to ignore, measured
BM25 scores a chunk higher when it contains the question's words, especially rare ones. We compare four versions on the development questions and keep the best one:

| Version | Words ignored | Reason to test it |
|---|---|---|
| A | none | the baseline |
| B | scikit-learn's standard English stopword list (318 words) | the usual choice in NLP |
| C | the same list, but keeping "not", "no", "nor" | a negation turns a rule around: *"shall **not** be considered high-risk"* (Article 6(3)) |
| D | C + the words the LLM judge and the reviewer marked IGNORE (step 3) | words that are frequent **in our chunks** |

In [ ]:
from rank_bm25 import BM25Okapi

NEGATIONS = {"not", "no", "nor"}
print("Chunks that contain a negation:", sum(bool(NEGATIONS & set(ws)) for ws in chunk_words), "of", len(chunks))

VERSIONS = {"A: none": set(),
            "B: standard stopwords": set(ENGLISH_STOP_WORDS),
            "C: stopwords, keep not/no/nor": set(ENGLISH_STOP_WORDS) - NEGATIONS,
            "D: C + LLM-approved words": (set(ENGLISH_STOP_WORDS) - NEGATIONS) | LLM_IGNORE}

def bm25_search_for(ignore):
    """A BM25 search function that ignores the given words."""
    index = BM25Okapi([[w for w in words_of(c["embed_text"]) if w not in ignore] for c in chunks])
    def search_fn(question):
        scores = index.get_scores([w for w in words_of(question) if w not in ignore])
        return [chunks[i]["chunk_id"] for i in np.argsort(-scores)]
    return search_fn

bm25_results = pd.DataFrame([{"version": name, **evaluate(bm25_search_for(ignore), RET["top_k"])}
                             for name, ignore in VERSIONS.items()])
BEST_BM25 = bm25_results.sort_values(["Recall@k", "MRR"], ascending=False).iloc[0]["version"]
print("Best on the development questions:", BEST_BM25)
bm25_results

**Comment:** _(1–2 sentences after running: did removing stopwords help? did keeping the negations or the LLM list change anything?)_

## 5. Embedding: measured choices
`bge-base-en-v1.5` (D10) turns each text into 768 numbers; texts with a similar meaning get similar numbers. We test two things the model card and common practice suggest, instead of assuming them:
1. **Full text vs text without stopwords.** The model was trained on normal sentences, so removing words may hurt.
2. **With vs without the instruction** *"Represent this sentence for searching relevant passages: "* in front of the question. The model card recommends it for short questions.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMB["model"], device="cpu")
stop = set(ENGLISH_STOP_WORDS) - NEGATIONS

def embed(texts):
    return model.encode(texts, batch_size=EMB["batch_size"], normalize_embeddings=True, show_progress_bar=False)

start = time.time()
full_vectors = embed([c["embed_text"] for c in chunks])
print("vectors:", full_vectors.shape, f"in {time.time() - start:.1f} s")
clean_vectors = embed([" ".join(w for w in words_of(c["embed_text"]) if w not in stop) for c in chunks])

def dense_search_for(vectors, instruction, remove_stopwords=False):
    """A dense search function over the given chunk vectors."""
    def search_fn(question):
        if remove_stopwords:
            question = " ".join(w for w in words_of(question) if w not in stop)
        q = embed([(EMB["query_instruction"] if instruction else "") + question])[0]
        return [chunks[i]["chunk_id"] for i in np.argsort(-(vectors @ q))]
    return search_fn

DENSE_VERSIONS = {"full text, with instruction": (full_vectors, True, False),
                  "full text, no instruction": (full_vectors, False, False),
                  "stopwords removed, with instruction": (clean_vectors, True, True)}
dense_results = pd.DataFrame([{"version": name, **evaluate(dense_search_for(*v), RET["top_k"])}
                              for name, v in DENSE_VERSIONS.items()])
BEST_DENSE = dense_results.sort_values(["Recall@k", "MRR"], ascending=False).iloc[0]["version"]
print("Best on the development questions:", BEST_DENSE)
dense_results

**Comment:** _(1–2 sentences after running: does the full text work better? does the instruction help?)_

## 6. Store the vectors in Chroma
Chroma (D11) keeps the vectors, the chunk texts and the metadata, and returns the closest chunks to a question vector, with cosine distance. It is saved in `data/vector-store/`; it is not committed to Git and is rebuilt by this notebook. We store the version that won in step 5.

In [ ]:
import chromadb

best_vectors, use_instruction, remove_stopwords = DENSE_VERSIONS[BEST_DENSE]
client = chromadb.PersistentClient(path=STORE["path"])
if STORE["collection"] in [c.name for c in client.list_collections()]:
    client.delete_collection(STORE["collection"])
collection = client.create_collection(STORE["collection"], metadata={"hnsw:space": STORE["distance"]})
META = ["ref", "title", "chapter", "area", "priority", "version_date", "prev_id", "next_id"]
collection.add(ids=[c["chunk_id"] for c in chunks], embeddings=best_vectors.tolist(),
               documents=[c["text"] for c in chunks], metadatas=[{k: str(c.get(k, "")) for k in META} for c in chunks])

def search_dense(question, n):
    """The n closest chunk IDs from Chroma."""
    if remove_stopwords:
        question = " ".join(w for w in words_of(question) if w not in stop)
    q = embed([(EMB["query_instruction"] if use_instruction else "") + question])[0]
    return collection.query(query_embeddings=[q.tolist()], n_results=n)["ids"][0]

print(collection.count(), "chunks in", STORE["path"], "| stored version:", BEST_DENSE)

## 7. Dense vs BM25 vs hybrid, measured
**Hybrid** merges the dense list and the BM25 list with Reciprocal Rank Fusion (D12): each chunk gets `1 / (k + rank)` from each list and the scores are added, so a chunk high in both lists comes first. k = 60 is the value from the RRF paper (Cormack, Clarke and Büttcher, 2009). The design document plans this as **E1 = dense (baseline)** and **E2 = hybrid**.

In [ ]:
best_bm25 = bm25_search_for(VERSIONS[BEST_BM25])

def search(question, mode=RET["mode"], k=None):
    """Ranked chunk IDs for a question; mode is 'dense', 'bm25' or 'hybrid'."""
    n = RET["candidates"]
    if mode == "dense":
        return search_dense(question, n)
    if mode == "bm25":
        return best_bm25(question)[:n]
    rrf = Counter()
    for ranked in (search_dense(question, n), best_bm25(question)[:n]):
        for rank, chunk_id in enumerate(ranked, start=1):
            rrf[chunk_id] += 1 / (RET["rrf_k"] + rank)
    return [chunk_id for chunk_id, _ in rrf.most_common()]

mode_results = pd.DataFrame([{"mode": m, **evaluate(lambda q, m=m: search(q, m), RET["top_k"])} for m in ["dense", "bm25", "hybrid"]])
mode_results

**Comment:** _(1–2 sentences after running: does hybrid beat dense on our questions?)_

## 8. How many chunks to return (top-k), measured
D12: choose k from 3, 5 and 8 by Recall@k. More chunks find more gold provisions, but give the answer model more text to read (slower answers, more noise).

In [ ]:
topk_results = pd.DataFrame([{"k": k, **evaluate(lambda q: search(q), k)} for k in [3, 5, 8]])
topk_results

**Comment:** _(1–2 sentences after running: where does Recall stop growing?)_

## 9. A handful of test questions (11/11 check)
Five questions that were **not** used for tuning, with the top 5 hybrid chunks next to the gold source. ✓ = a gold provision.

In [ ]:
rows = []
for qid in ["BM-01", "BM-03", "BM-07", "EM-09", "EM-10"]:
    q = all_questions[qid]
    found = search(q["question"])[:RET["top_k"]]
    rows.append({"id": qid, "question": q["question"][:100] + "...",
                 "top 5 (hybrid)": "\n".join(("✓ " if any(same_provision(ref_of[i], g) for g in gold_parts(q)) else "· ") + ref_of[i] for i in found),
                 "gold source": q["gold_source"]})
pd.DataFrame(rows).style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})

**Comment:** _(1–2 sentences after running)_

## 10. The decisions and their evidence
This table is built from the results above, so it always shows the numbers of the last run. Copy the chosen settings into `config/rag.yaml` if they differ from it.

In [ ]:
best_mode = mode_results.sort_values(["Recall@k", "MRR"], ascending=False).iloc[0]
pd.DataFrame([
    {"decision": "Words BM25 ignores", "choice": BEST_BM25, "evidence": bm25_results.to_dict("records")},
    {"decision": "Text for the embedding model", "choice": BEST_DENSE, "evidence": dense_results.to_dict("records")},
    {"decision": "Search mode", "choice": best_mode["mode"], "evidence": mode_results.to_dict("records")},
    {"decision": "Top-k", "choice": "see step 8", "evidence": topk_results.to_dict("records")},
    {"decision": "Words the LLM judge marked IGNORE (after review)", "choice": sorted(LLM_IGNORE), "evidence": f"{len(judge_table)} candidates, judge {JUDGE['model']}"},
])

**Limits.** Six development questions are few: one question changes Recall by about 0.17. These results choose the settings for the baseline; the final comparison is run on the whole `val` split in KAN-99, logged in W&B (instructor's slides: *"Log hyperparameters (prompt, chunk size, model) and compare runs"*). Answer metrics such as Faithfulness need the generated answers and come in Sprint 3.